### Import dependencies

In [ ]:
pip install opencv-python pandas numpy tqdm

In [21]:
from pathlib import Path
from datetime import datetime
import re
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import random

/Users/alopias/Desktop/Huyen-deePi/deePi-laptop/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Establish paths and some settings

In [34]:
ROOT = Path(r"/Users/alopias/Desktop/deePi-video-processing/converted-videos") 
OUT = Path(r"/Users/alopias/Desktop/Huyen-deePi/deePi-laptop")

BLACK_FRAME_DIR = OUT / "black_frames"
OUT.mkdir(parents=True, exist_ok=True)
BLACK_FRAME_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Settings
# -------------------------
TARGET_BLACK_FRAMES = 150
FPS = 26
MIN_SECONDS_APART = 10
MAX_FRAMES_PER_VID = 6
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
VIDEO_RE = re.compile(r"^\d{12}\.mp4$")

Black frame threshold settings 

In [47]:
BLACK_RULE = {
    "mean_brightness_max": 8.0,
    "median_brightness_max": 6.0,
    "pct_pixels_above_20_max": 0.01,    # <= 1% pixels brighter than 20
    "pct_pixels_above_40_max": 0.002,   # <= 0.2% pixels brighter than 40
    "entropy_max": 3.5,
}

### Get metadata, construct frame index table, compute frame metrics, etc. (helper functions)

In [48]:
def parse_video_datetime(video_path):
    """
    Parses filename like 202306222131.mp4
    Into: year=2023, month=6, day=22, hour=21, minute=31
    """
    dt = datetime.strptime(video_path.stem, "%Y%m%d%H%M")
    return {"year": dt.year, "month": dt.month, "day": dt.day, "hour": dt.hour, "minute": dt.minute, "datetime": dt}

In [49]:
def build_video_table(root):
    """
    Builds an internal table of videos.
    This is only used inside Python.
    It is NOT saved as an output CSV.
    """
    rows = []
    for camera_folder in sorted(root.iterdir()):
        if not camera_folder.is_dir():
            continue
        camera = camera_folder.name
        for video_path in sorted(camera_folder.glob("*.mp4")):
            if not VIDEO_RE.match(video_path.name):
                continue
            dt_info = parse_video_datetime(video_path)
            rows.append({
                "camera": camera,
                "video": video_path.name,
                "video_path": video_path,
                **dt_info})
    return pd.DataFrame(rows)


In [50]:
def compute_frame_metrics(frame_bgr):
    """
    Computes brightness metrics for one frame.
    """
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    mean_brightness = float(gray.mean())
    median_brightness = float(np.median(gray))
    max_brightness = int(gray.max())

    pct_pixels_above_20 = float((gray > 20).mean())
    pct_pixels_above_40 = float((gray > 40).mean())
    pct_pixels_above_80 = float((gray > 80).mean())

    # Entropy: flat black images have low entropy.
    hist = np.bincount(gray.ravel(), minlength=256)
    probs = hist[hist > 0] / gray.size
    entropy = float(-(probs * np.log2(probs)).sum())

    return {
        "mean_brightness": mean_brightness,
        "median_brightness": median_brightness,
        "max_brightness": max_brightness,
        "pct_pixels_above_20": pct_pixels_above_20,
        "pct_pixels_above_40": pct_pixels_above_40,
        "pct_pixels_above_80": pct_pixels_above_80,
        "entropy": entropy,
    }

In [51]:
def is_black_frame(metrics):
    """
    Decide if frame is black based on the black frame rules 
    """
    return (
        metrics["mean_brightness"] <= BLACK_RULE["mean_brightness_max"]
        and metrics["median_brightness"] <= BLACK_RULE["median_brightness_max"]
        and metrics["pct_pixels_above_20"] <= BLACK_RULE["pct_pixels_above_20_max"]
        and metrics["pct_pixels_above_40"] <= BLACK_RULE["pct_pixels_above_40_max"]
        and metrics["entropy"] <= BLACK_RULE["entropy_max"]
    )


In [52]:
def sample_frame_indices(total_frames, fps):
    """
    Randomly samples candidate frame indices from a video. The selected candidate frames are at least MIN_SECONDS_APART apart.
    """
    if total_frames <= 0:
        return []

    min_gap = int(round(fps * MIN_SECONDS_APART))
    if min_gap <= 0:
        min_gap = FPS * MIN_SECONDS_APART

    # Random offset prevents always checking frame 0, 260, 520, etc.
    max_offset = min(min_gap - 1, max(total_frames - 1, 0))
    offset = random.randint(0, max_offset)
    possible_frames = list(range(offset, total_frames, min_gap))
    if len(possible_frames) == 0:
        return []
    n_to_sample = min(MAX_FRAMES_PER_VID, len(possible_frames))
    sampled = random.sample(possible_frames, k=n_to_sample)
    return sorted(sampled)

In [53]:
def make_balanced_video_order(camera_df):
    """
    Creates a randomized video order that spreads videos across different hours to avoid sampling from same time every day.
    """
    groups = {}
    for hour, sub_df in camera_df.groupby("hour"):
        records = sub_df.to_dict("records")
        random.shuffle(records)
        groups[hour] = records
    ordered = []
    while groups:
        hours = list(groups.keys())
        random.shuffle(hours)
        for hour in hours:
            if hour not in groups:
                continue
            if len(groups[hour]) == 0:
                del groups[hour]
                continue
            ordered.append(groups[hour].pop())
            if len(groups[hour]) == 0:
                del groups[hour]
    return ordered

In [54]:
def make_camera_targets(camera_names, total_target):
    """
    Splits the total target approximately evenly across cameras.
    Example: 160 frames across 3 cameras -> 54, 53, 53
    """
    base = total_target // len(camera_names)
    remainder = total_target % len(camera_names)
    targets = {}
    for i, camera in enumerate(camera_names):
        targets[camera] = base + (1 if i < remainder else 0)
    return targets

### Build internal video list

In [55]:
video_df = build_video_table(ROOT)
if len(video_df) == 0:
    raise RuntimeError("No videos found. Check ROOT path and filename format.")
camera_names = sorted(video_df["camera"].unique())
print("Total videos found:", len(video_df))
print("Cameras found:", camera_names)
print()
print(video_df["camera"].value_counts())

Total videos found: 8187
Cameras found: ['10.0.11.2', '10.0.12.2', '10.0.16.2']

camera
10.0.12.2    2730
10.0.11.2    2729
10.0.16.2    2728
Name: count, dtype: int64


### Extract black frames

In [56]:
camera_targets = make_camera_targets(camera_names, TARGET_BLACK_FRAMES)
print()
print("Target black frames per camera:")
for camera, target in camera_targets.items():
    print(f"  {camera}: {target}")

black_rows = []
saved_count_by_camera = {camera: 0 for camera in camera_names}
global_black_frame_id = 1

for camera in camera_names:
    camera_df = video_df[video_df["camera"] == camera].copy()
    video_order = make_balanced_video_order(camera_df)
    target_for_camera = camera_targets[camera]
    progress = tqdm(video_order, total=len(video_order),desc=f"Searching {camera}")
    for row in progress:
        if saved_count_by_camera[camera] >= target_for_camera:
            break
        video_path = row["video_path"]
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            continue
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps is None or fps <= 0:
            fps = FPS
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_indices = sample_frame_indices(
            total_frames=total_frames,
            fps=fps,
        )
        for frame_index in frame_indices:
            if saved_count_by_camera[camera] >= target_for_camera:
                break
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
            ok, frame = cap.read()
            if not ok or frame is None:
                continue
            metrics = compute_frame_metrics(frame)
            if is_black_frame(metrics):
                image_name = (
                    f"black_{global_black_frame_id:04d}__"
                    f"{row['camera']}__"
                    f"{row['video'].replace('.mp4', '')}__"
                    f"frame_{frame_index:06d}.png"
                )
                image_path = BLACK_FRAME_DIR / image_name
                saved_ok = cv2.imwrite(str(image_path), frame)
                if not saved_ok:
                    print(f"Warning: failed to save {image_path}")
                    continue

                black_rows.append({
                    "black_frame_id": global_black_frame_id,
                    "camera": row["camera"],
                    "video": row["video"],
                    "year": row["year"],
                    "month": row["month"],
                    "day": row["day"],
                    "datetime": row["datetime"],
                    "frame_index": frame_index,
                    "fps": fps,
                    "total_frames": total_frames,
                    "image_name": image_name,
                    "image_path": str(image_path),
                    **metrics,
                })
                saved_count_by_camera[camera] += 1
                global_black_frame_id += 1
                progress.set_postfix({
                    "saved": saved_count_by_camera[camera],
                    "target": target_for_camera,
                })
        cap.release()


Target black frames per camera:
  10.0.11.2: 50
  10.0.12.2: 50
  10.0.16.2: 50


Searching 10.0.16.2:   2%|▏         | 50/2728 [00:24<21:49,  2.05it/s, saved=50, target=50]


### Save final csv table

In [57]:
black_df = pd.DataFrame(black_rows)
# Keep only the first TARGET_BLACK_FRAMES just in case
black_df = black_df.head(TARGET_BLACK_FRAMES).copy()
# Re-number cleanly from 1 to N
if len(black_df) > 0:
    black_df["black_frame_id"] = range(1, len(black_df) + 1)
black_csv = OUT / "black_frame_manifest.csv"
black_df.to_csv(black_csv, index=False)

print()
print("Done.")
print("Saved black frames:", len(black_df))
print("Images saved in:", BLACK_FRAME_DIR)
print("Manifest saved to:", black_csv)
print()
print("Saved by camera:")
print(black_df["camera"].value_counts() if len(black_df) > 0 else "No frames saved.")

if len(black_df) < TARGET_BLACK_FRAMES:
    print()
    print("Warning: fewer black frames were found than requested.")
    print("Try relaxing BLACK_RULE thresholds near the top of the notebook.")
    print("For example:")
    print("  mean_brightness_max: 10 or 12")
    print("  median_brightness_max: 8 or 10")
    print("  pct_pixels_above_20_max: 0.02")
    print("  entropy_max: 4.0")
black_df.head()


Done.
Saved black frames: 150
Images saved in: /Users/alopias/Desktop/Huyen-deePi/deePi-laptop/black_frames
Manifest saved to: /Users/alopias/Desktop/Huyen-deePi/deePi-laptop/black_frame_manifest.csv

Saved by camera:
camera
10.0.11.2    50
10.0.12.2    50
10.0.16.2    50
Name: count, dtype: int64


,black_frame_id,camera,video,year,month,day,datetime,frame_index,fps,total_frames,image_name,image_path,mean_brightness,median_brightness,max_brightness,pct_pixels_above_20,pct_pixels_above_40,pct_pixels_above_80,entropy
0,1,10.0.11.2,202309100548.mp4,2023,9,10,2023-09-10 05:48:00,187,1078.279129,1799,black_0001__10.0.11.2__202309100548__frame_000...,/Users/alopias/Desktop/Huyen-deePi/deePi-lapto...,0.000032,0.0,2,0.0,0.0,0.0,0.000525
1,2,10.0.11.2,202307210346.mp4,2023,7,21,2023-07-21 03:46:00,323,1078.279129,1799,black_0002__10.0.11.2__202307210346__frame_000...,/Users/alopias/Desktop/Huyen-deePi/deePi-lapto...,0.000028,0.0,4,0.0,0.0,0.0,0.000421
2,3,10.0.11.2,202310261546.mp4,2023,10,26,2023-10-26 15:46:00,1381,1078.279129,1799,black_0003__10.0.11.2__202310261546__frame_001...,/Users/alopias/Desktop/Huyen-deePi/deePi-lapto...,0.000003,0.0,1,0.0,0.0,0.0,0.000057
3,4,10.0.11.2,202309111746.mp4,2023,9,11,2023-09-11 17:46:00,1620,1078.279129,1799,black_0004__10.0.11.2__202309111746__frame_001...,/Users/alopias/Desktop/Huyen-deePi/deePi-lapto...,0.002723,0.0,17,0.0,0.0,0.0,0.025115
4,5,10.0.11.2,202308240646.mp4,2023,8,24,2023-08-24 06:46:00,1228,1078.279129,1799,black_0005__10.0.11.2__202308240646__frame_001...,/Users/alopias/Desktop/Huyen-deePi/deePi-lapto...,0.000001,0.0,1,0.0,0.0,0.0,0.000030
